<a href="https://colab.research.google.com/github/Toan-ten/Arxiv-Project-Group10/blob/feat%2Fs2-request-optimization/notebooks/m1_pipeline_thu_thap_va_kiem_toan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Mở trong Colab"/></a>

# Quy trình M1: thu thập, đối soát và xuất dữ liệu

Notebook thực hiện một quy trình trên danh sách ID của thành viên: cấu hình phạm vi → thu thập arXiv → thu thập Semantic Scholar → kiểm toán và xuất bảng. Mỗi pha có điểm khôi phục riêng.

Chạy các ô theo thứ tự. Ô cấu hình yêu cầu Colab CPU, MSSV hợp lệ và hai Secret `KAGGLE_API_TOKEN`, `SEMANTIC_SCHOLAR_API_KEY`. Đặt `COHORT_LIMIT = 1` để chạy thử 100 ID; đặt `None` để chạy đủ 5.000 ID. Phạm vi chạy thử và toàn dải dùng thư mục Drive riêng.

Nếu ô báo lỗi hoặc bị ngắt, đọc thông báo và đường dẫn checkpoint. Sau khi xác nhận không còn phiên khác ghi cùng phạm vi, chạy lại ô tương ứng với cùng MSSV, giới hạn và thư mục Drive. Không chạy kiểm toán khi hai pha thu thập chưa hoàn tất.

## Ô 1 — Cấu hình Colab và phạm vi

Ô này kiểm tra CPU, MSSV và hai Secret; gắn Drive; tải repo, chuyển tới SHA runtime đã ghim; cài gói cần thiết; rồi in phạm vi ID, dấu vân tay và đường dẫn lưu.

Nếu báo kernel đã nạp `src`, khởi động lại runtime rồi chạy ô này trước. Nếu báo thiếu Secret, thêm Secret tương ứng trong Colab và cấp quyền truy cập. Nếu MSSV hoặc SHA không hợp lệ, sửa cấu hình rồi chạy lại. Lỗi gắn Drive, tải mã hoặc cài gói cần được xử lý trước khi chạy các pha.

In [ ]:
# SHA mã nguồn dùng cho quy trình M1.
import os
from pathlib import Path
import re
import subprocess
import sys

M1_RUNTIME_COMMIT = "e29ea3c9a0525371b4ee6fb6b60ba069ac00b647"
MSSV = ""  # Nhập MSSV trong config/group10_id_ranges.json.
COHORT_LIMIT = None  # None: 50 cohort; 1..50: số cohort đầu tiên.
PILOT_ROOT = COHORT_LIMIT is not None
DRIVE_BASE = Path("/content/drive/MyDrive/M1-pipeline-optimization-pilot" if PILOT_ROOT else "/content/drive/MyDrive/M1-pipeline-optimization-full")
LOCAL_BASE = Path("/content/m1-pipeline-optimization-pilot-local" if PILOT_ROOT else "/content/m1-pipeline-optimization-local")

loaded_src = sorted(name for name in sys.modules if name == "src" or name.startswith("src."))
if loaded_src:
    raise RuntimeError("Kernel đã nạp src trước khi checkout. Chọn Runtime > Restart session rồi chạy lại cell này.")
if os.environ.get("COLAB_GPU") == "1" or os.environ.get("COLAB_TPU_ADDR"):
    raise RuntimeError("Chọn runtime chỉ dùng CPU trong Runtime settings.")
if not MSSV.strip():
    raise ValueError("Nhập MSSV trước khi tiếp tục.")

try:
    from google.colab import userdata
except Exception:
    raise RuntimeError("Notebook M1 cần chạy trên Colab để đọc Colab Secrets.") from None


def _read_required_m1_secret(name):
    try:
        value = userdata.get(name)
    except Exception:
        raise RuntimeError(
            f"Không đọc được Colab Secret `{name}`. Hãy cấp quyền truy cập Secret cho notebook."
        ) from None
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"Thiếu hoặc rỗng Colab Secret bắt buộc `{name}`.")
    return value.strip()


for _secret_name in ("KAGGLE_API_TOKEN", "SEMANTIC_SCHOLAR_API_KEY"):
    _secret_value = _read_required_m1_secret(_secret_name)
del _secret_name, _secret_value

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = Path("/content/Arxiv-Project-Group10")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/Toan-ten/Arxiv-Project-Group10.git", str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if not re.fullmatch(r"[0-9a-f]{40}", M1_RUNTIME_COMMIT):
    raise RuntimeError("M1_RUNTIME_COMMIT phải có đủ 40 ký tự SHA.")
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", "--detach", M1_RUNTIME_COMMIT], check=True)
current_head = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
if current_head != M1_RUNTIME_COMMIT:
    raise RuntimeError("Mã runtime không khớp SHA đã ghim.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

from src.m1_workflow import build_member_selection
SELECTION = build_member_selection(MSSV, COHORT_LIMIT)
DRIVE_ROOT = DRIVE_BASE / MSSV
LOCAL_ROOT = LOCAL_BASE / MSSV
print(f"Runtime: {current_head}")
print("Chiến lược S2: batch_m1")
print(f"MSSV: {MSSV}; IDs: {SELECTION['expected_count']}; cohorts: {len(SELECTION['cohorts'])}")
print(f"Selection fingerprint: {SELECTION['fingerprint']}")
print(f"Drive root: {DRIVE_ROOT}; local root: {LOCAL_ROOT}")


## Ô 2 — Thu thập arXiv

Ô này xác minh ID bằng Atom, lập danh sách các lần nộp và tải mã nguồn. Pha yêu cầu `KAGGLE_API_TOKEN` cùng snapshot và index Kaggle hợp lệ. Kết quả và điểm khôi phục được lưu theo phạm vi đã chọn.

HTTP 429 có thể được tiếp tục tự động trong giới hạn đã cấu hình. HTTP 403, lỗi Kaggle, lỗi điểm khôi phục hoặc hết ngân sách làm ô dừng. Sau khi xử lý nguyên nhân, chạy lại ô với cùng cấu hình và thư mục Drive.

In [ ]:
from src.m1_workflow import run_m1_arxiv_member

KAGGLE_API_TOKEN = _read_required_m1_secret("KAGGLE_API_TOKEN")
try:
    ARXIV_RESULT = run_m1_arxiv_member(
        MSSV,
        drive_root=DRIVE_ROOT,
        local_root=LOCAL_ROOT,
        cohort_limit=COHORT_LIMIT,
        kaggle_token=KAGGLE_API_TOKEN,
        auto_resume_429=True,
        auto_resume_429_max_per_cohort=1,
        auto_resume_429_max_per_session=2,
        auto_resume_429_cooldown_min_seconds=300,
        auto_resume_429_cooldown_max_seconds=600,
    )
finally:
    KAGGLE_API_TOKEN = None

print({key: ARXIV_RESULT.get(key) for key in (
    "stage", "selection_fingerprint", "expected_id_count",
    "source_complete_papers", "stopped", "stop_reason", "m1_complete", "kaggle", "auto_resume_429",
)})


## Ô 3 — Thu thập Semantic Scholar

Ô này lấy metadata và references cho cùng danh sách ID bằng `batch_m1`. Client kiểm tra định danh và số lượng; bài chưa đủ bằng chứng được tiếp tục bằng phân trang. References đang thu thập dở cũng được tiếp tục bằng phân trang. Ba trường quan hệ tùy chọn `contexts`, `intents`, `isInfluential` không thuộc phạm vi dữ liệu M1.

Pha yêu cầu `SEMANTIC_SCHOLAR_API_KEY`. HTTP 429 có thể được tiếp tục tự động trong giới hạn đã cấu hình. HTTP 403, lỗi xác minh checkpoint hoặc hết ngân sách làm ô dừng. Khi phân trang kết thúc nhưng số lượng vẫn lệch, dữ liệu được giữ và cảnh báo được ghi vào `quality.json`.

In [ ]:
from src.m1_workflow import run_m1_s2_member

S2_API_KEY = _read_required_m1_secret("SEMANTIC_SCHOLAR_API_KEY")
try:
    S2_RESULT = run_m1_s2_member(
        MSSV,
        drive_root=DRIVE_ROOT,
        local_root=LOCAL_ROOT,
        semantic_scholar_api_key=S2_API_KEY,
        cohort_limit=COHORT_LIMIT,
        reference_strategy="batch_m1",
        auto_resume_429=True,
        auto_resume_429_max_per_cohort=1,
        auto_resume_429_max_per_session=2,
        auto_resume_429_cooldown_min_seconds=300,
        auto_resume_429_cooldown_max_seconds=600,
    )
finally:
    S2_API_KEY = None

S2_AUTO_RESUME = S2_RESULT.get("auto_resume_429", {})
S2_RESULT_SUMMARY = {key: S2_RESULT.get(key) for key in (
    "stage", "selection_fingerprint", "expected_id_count",
    "metadata_complete", "references_complete", "stopped", "stop_reason", "m1_complete",
)}
S2_RESULT_SUMMARY["auto_resume_429"] = {key: S2_AUTO_RESUME.get(key) for key in (
    "resume_attempts", "cooldown_seconds", "events_path", "events_sha256",
)}
S2_RESULT_SUMMARY["reference_quality"] = {
    key: value for key, value in S2_RESULT.get("reference_quality", {}).items() if key != "papers"
}
print(S2_RESULT_SUMMARY)


## Ô 4 — Kiểm toán, đối soát và xuất bảng

Chạy ô này sau khi hai pha thu thập đã ghi đủ bằng chứng. Kiểm toán xác minh phạm vi, định danh, xuất xứ, checksum và trạng thái references; sau đó hợp nhất dữ liệu và xuất bảng nếu đạt điều kiện.

Nếu `is_reconciled` không đúng hoặc `export_status` không phải `exported`, xem báo cáo và xử lý lỗi được nêu. Kiểm toán bổ sung ngày references còn thiếu từ bằng chứng arXiv đã xác minh; ngày không tìm được được giữ là null và ghi lý do. Không chạy lại S2 chỉ vì cảnh báo count.

In [ ]:
from src.m1_workflow import run_m1_audit_member

AUDIT_RESULT = run_m1_audit_member(
    MSSV,
    drive_root=DRIVE_ROOT,
    local_root=LOCAL_ROOT,
    cohort_limit=COHORT_LIMIT,
)
print({key: AUDIT_RESULT.get(key) for key in (
    "selection_fingerprint", "expected_id_count", "merged_s2_ids",
    "is_reconciled", "integrity_failed_papers", "unprocessed_papers",
    "export_status", "export_error", "report_path", "benchmark_summary_path",
)})

print("Chất lượng S2:", {key: value for key, value in AUDIT_RESULT.get("reference_quality", {}).items() if key != "papers"})
print("Bổ sung ngày:", {key: value for key, value in (AUDIT_RESULT.get("reference_dates") or {}).items() if key not in {"dates", "entries"}})


## Cách chạy lại và giới hạn kết quả

Nếu một pha dừng, giữ nguyên MSSV, `COHORT_LIMIT` và Drive root rồi chạy lại đúng ô đó. Nếu chạy S2 trước arXiv, hoàn tất ô arXiv trước khi kiểm toán. Không để hai runtime cùng ghi một phạm vi.

Lượt 100 ID chỉ kiểm tra vận hành. Phép đo toàn nhóm cần bao phủ đủ 20.000 vị trí và ghi nhận thời gian chờ API; không suy rộng kết quả lượt thử ra toàn nhóm. Xem [hướng dẫn thành viên](../docs/M1_COLAB_MEMBER_GUIDE.md), [quy trình M1](../docs/operations/m1_production.md) và [hướng dẫn S2](../docs/guides/s2_integration.md).